# Token Embedding Analysis

The embedding matrix W_E and unembedding matrix W_U are where the model interfaces
with the vocabulary. Analyzing their structure reveals which tokens the model treats
as similar, how it organizes semantic space, and whether it reuses representations.

This notebook covers the `irtk.embeddings` module.

## Why This Matters

The embedding space defines the model's initial representation of tokens. Analyzing embedding geometry — nearest neighbors, analogies, and alignment with the unembedding — reveals what semantic structure the model starts with before any attention or MLP processing.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import embeddings

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Vocabulary size: {model.cfg.d_vocab}")
print(f"Embedding dimension: {model.cfg.d_model}")

## 1. Embedding Similarity

Which tokens have similar embeddings? This reveals the model's notion of token similarity.

In [ ]:
# Compare related tokens
pairs = [
    (" cat", " dog"),
    (" cat", " cats"),
    (" happy", " sad"),
    (" happy", " joyful"),
    (" Paris", " France"),
    (" Paris", " pizza"),
]

for word_a, word_b in pairs:
    tok_a = tokenizer.encode(word_a)[0]
    tok_b = tokenizer.encode(word_b)[0]
    sim = embeddings.embedding_similarity(model, tok_a, tok_b)
    print(f"{word_a:>10s} vs {word_b:<10s}: {sim:.4f}")

In [ ]:
# Compare embed vs unembed space
for word_a, word_b in pairs[:3]:
    tok_a = tokenizer.encode(word_a)[0]
    tok_b = tokenizer.encode(word_b)[0]
    sim_e = embeddings.embedding_similarity(model, tok_a, tok_b, space="embed")
    sim_u = embeddings.embedding_similarity(model, tok_a, tok_b, space="unembed")
    print(f"{word_a:>10s} vs {word_b:<10s}:  embed={sim_e:.4f}  unembed={sim_u:.4f}")

## 2. Nearest Neighbors

Find the tokens closest to a given token or direction in embedding space.

In [ ]:
# Nearest neighbors of " king"
query_token = tokenizer.encode(" king")[0]
neighbors = embeddings.nearest_neighbors(model, query_token, k=10)
print("Nearest neighbors of ' king':")
for tid, sim in neighbors:
    print(f"  {tokenizer.decode([tid]):>15s} ({tid:5d}): {sim:.4f}")

In [ ]:
# Nearest neighbors in unembed space
neighbors_u = embeddings.nearest_neighbors(model, query_token, k=10, space="unembed")
print("Nearest neighbors of ' king' (unembed space):")
for tid, sim in neighbors_u:
    print(f"  {tokenizer.decode([tid]):>15s} ({tid:5d}): {sim:.4f}")

## 3. PCA of Embedding Space

Visualize how tokens are distributed in embedding space using PCA.

In [ ]:
# PCA on a curated set of tokens
words = [" cat", " dog", " fish", " bird",  # animals
         " red", " blue", " green", " yellow",  # colors
         " one", " two", " three", " four",  # numbers
         " happy", " sad", " angry", " calm"]  # emotions
token_ids = [tokenizer.encode(w)[0] for w in words]

result = embeddings.embedding_pca(model, token_ids=token_ids, n_components=2)
proj = result["projections"]
print(f"Explained variance: {result['explained_variance']}")

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['red'] * 4 + ['blue'] * 4 + ['green'] * 4 + ['purple'] * 4
ax.scatter(proj[:, 0], proj[:, 1], c=colors, s=100)
for i, word in enumerate(words):
    ax.annotate(word.strip(), (proj[i, 0], proj[i, 1]), fontsize=10,
               textcoords="offset points", xytext=(5, 5))
ax.set_xlabel(f"PC1 ({result['explained_variance'][0]:.1%})")
ax.set_ylabel(f"PC2 ({result['explained_variance'][1]:.1%})")
ax.set_title("Token Embeddings PCA")
ax.legend(['Animals', 'Colors', 'Numbers', 'Emotions'], loc='best')
plt.tight_layout()
plt.show()

## 4. Token Analogies

Solve analogies like "man is to woman as king is to ???" using vector arithmetic.

In [ ]:
# man:woman :: king:?
a = tokenizer.encode(" man")[0]
b = tokenizer.encode(" woman")[0]
c = tokenizer.encode(" king")[0]

results = embeddings.token_analogy(model, a, b, c, k=5)
print("man : woman :: king : ?")
for tid, sim in results:
    print(f"  {tokenizer.decode([tid]):>15s}: {sim:.4f}")

In [ ]:
# France:Paris :: Germany:?
a = tokenizer.encode(" France")[0]
b = tokenizer.encode(" Paris")[0]
c = tokenizer.encode(" Germany")[0]

results = embeddings.token_analogy(model, a, b, c, k=5)
print("France : Paris :: Germany : ?")
for tid, sim in results:
    print(f"  {tokenizer.decode([tid]):>15s}: {sim:.4f}")

## 5. Embedding Clustering

Group tokens by embedding similarity to discover how the model organizes its vocabulary.

In [ ]:
# Cluster a set of tokens
cluster_words = [
    " cat", " dog", " fish",  # animals
    " red", " blue", " green",  # colors
    " one", " two", " three",  # numbers
    " run", " walk", " jump",  # verbs
]
cluster_ids = [tokenizer.encode(w)[0] for w in cluster_words]

result = embeddings.embedding_cluster(model, cluster_ids, n_clusters=4)
for cluster_idx in range(4):
    members = [cluster_words[i] for i in range(len(cluster_words)) if result["labels"][i] == cluster_idx]
    print(f"Cluster {cluster_idx}: {', '.join(m.strip() for m in members)}")

## 6. Embed-Unembed Alignment

Does the model use the same representation for reading (W_E) and writing (W_U) tokens?
High alignment means the model reuses representations; low alignment means they diverge.

In [ ]:
# Measure alignment for all tokens
alignment = embeddings.embed_unembed_alignment(model)
print(f"Mean alignment: {np.mean(alignment):.4f}")
print(f"Std alignment:  {np.std(alignment):.4f}")
print(f"Min alignment:  {np.min(alignment):.4f}")
print(f"Max alignment:  {np.max(alignment):.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(alignment, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel("Cosine Similarity (W_E row vs W_U column)")
ax.set_ylabel("Count")
ax.set_title("Embed-Unembed Alignment Distribution")
ax.axvline(np.mean(alignment), color='red', linestyle='--', label=f'Mean={np.mean(alignment):.3f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Most and least aligned tokens
sorted_idx = np.argsort(alignment)
print("Most aligned tokens (W_E ~ W_U):")
for idx in sorted_idx[-10:][::-1]:
    print(f"  {tokenizer.decode([idx]):>15s}: {alignment[idx]:.4f}")

print("\nLeast aligned tokens:")
for idx in sorted_idx[:10]:
    print(f"  {tokenizer.decode([idx]):>15s}: {alignment[idx]:.4f}")

## Summary

| Function | Purpose |
|----------|--------|
| `embedding_similarity()` | Cosine similarity between two token embeddings |
| `nearest_neighbors()` | Find k nearest tokens to a token or direction |
| `embedding_pca()` | PCA of token embedding space |
| `token_analogy()` | Solve analogies using vector arithmetic |
| `embedding_cluster()` | Cluster tokens by embedding similarity |
| `embed_unembed_alignment()` | Measure W_E / W_U alignment per token |